<a href="https://colab.research.google.com/github/jeolin/BCCE_Experiment/blob/main/randomized_spectral_data_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Research on Copper Sulfate Absorbance Spectrum

Copper(II) sulfate solutions appear blue because they absorb light in the red-orange region of the visible spectrum and transmit blue-green light. The hydrated copper(II) ion, [Cu(H₂O)₆]²⁺, exhibits a characteristic broad absorption band due to d-d electronic transitions.

*   **Expected Shape**: The spectrum typically shows a broad absorption peak.
*   **Lambda Max (Absorbance Maximum)**: For aqueous copper(II) sulfate, the absorption maximum ($\lambda_{max}$) is generally found in the range of **750 nm to 820 nm**, corresponding to the lowest energy d-d transition. The exact position can vary slightly with concentration and solvent.

I will now generate a synthetic spectrum reflecting these properties.

In [7]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML
import random

# --- Define the fixed, broad wavelength range for underlying data ---
wavelengths_data = np.linspace(200, 1200, 1000)

# Generate lambda_max once when the notebook starts
initial_lambda_max = random.randint(795, 820)

# Global variables to store spectrum data for linking
_global_wavelengths_data = None
_global_absorbance_data = None
_global_max_absorbance = 0.0

def plot_copper_sulfate_spectrum(
    molar_absorptivity_input=1.0,
    min_wavelength_plot=350,
    max_wavelength_plot=1100,
    min_absorbance_plot=0.0,
    max_absorbance_plot=1.0,
    pathlength_input=1.0,
    concentration_input=1.0,
    marker_wavelength_input=350,
    fixed_lambda_max=None # Pass the fixed lambda_max
):
    global _global_wavelengths_data, _global_absorbance_data, _global_max_absorbance

    lambda_max = fixed_lambda_max # Use the pre-generated fixed value

    # Validate and set molar_absorptivity (epsilon)
    try:
        molar_absorptivity = float(molar_absorptivity_input) if molar_absorptivity_input else 1.0
        if molar_absorptivity <= 0:
            molar_absorptivity = 1.0
    except ValueError:
        molar_absorptivity = 1.0

    # Validate and set pathlength
    try:
        pathlength = float(pathlength_input) if pathlength_input else 1.0
        if pathlength <= 0:
            pathlength = 1.0
    except ValueError:
        pathlength = 1.0

    # Validate and set concentration
    try:
        concentration = float(concentration_input) if concentration_input else 1.0
        if concentration <= 0:
            concentration = 1.0
    except ValueError:
        concentration = 1.0

    # Validate plot limits
    if not isinstance(min_wavelength_plot, (int, float)) or not isinstance(max_wavelength_plot, (int, float)):
        min_wavelength_plot = 350
        max_wavelength_plot = 1100
    if min_wavelength_plot >= max_wavelength_plot:
        min_wavelength_plot = 350
        max_wavelength_plot = 1100

    # Validate y-axis plot limits
    if not isinstance(min_absorbance_plot, (int, float)) or not isinstance(max_absorbance_plot, (int, float)):
        min_absorbance_plot = 0.0
        max_absorbance_plot = 1.0
    if min_absorbance_plot >= max_absorbance_plot:
        min_absorbance_plot = 0.0
        max_absorbance_plot = 1.0

    # Validate marker_wavelength_input
    try:
        marker_wavelength = int(marker_wavelength_input) if marker_wavelength_input else 350
        marker_wavelength = max(min(marker_wavelength, 1200), 200)
    except ValueError:
        marker_wavelength = 350

    # Ensure plot limits are within the data generation range
    min_wavelength_plot = max(min_wavelength_plot, min(wavelengths_data))
    max_wavelength_plot = min(max_wavelength_plot, max(wavelengths_data))

    # Generate a synthetic absorbance spectrum using a Gaussian-like distribution
    bandwidth = 70
    gaussian_shape = np.exp(-(wavelengths_data - lambda_max)**2 / (2 * bandwidth**2))
    absorbance_data = molar_absorptivity * pathlength * concentration * gaussian_shape
    absorbance_data += 0.05 * molar_absorptivity * pathlength * concentration

    # Store data in global variables
    _global_wavelengths_data = wavelengths_data
    _global_absorbance_data = absorbance_data
    _global_max_absorbance = np.max(absorbance_data) if len(absorbance_data) > 0 else 0.0

    # Create the plot
    plt.figure(figsize=(5.0, 3.0))
    plt.plot(wavelengths_data, absorbance_data, color='blue')
    plt.title(rf'Absorbance Spectrum of Copper Sulfate (Curve $\lambda_{{max}}$: {lambda_max} nm)')
    plt.ylabel('Absorbance')

    # Mark the user-controlled marker wavelength on the plot
    plt.axvline(x=marker_wavelength, color='red', linestyle='--', label=rf'Marker $\lambda$ = {marker_wavelength} nm')
    text_y_pos = min(np.max(absorbance_data), max_absorbance_plot * 0.8)
    if text_y_pos > min_absorbance_plot and text_y_pos < max_absorbance_plot:
        plt.text(marker_wavelength + 10, text_y_pos, rf'Marker $\lambda$ = {marker_wavelength} nm', color='red')

    # Apply the user-defined x-axis and y-axis limits
    plt.xlim(min_wavelength_plot, max_wavelength_plot)
    plt.ylim(min_absorbance_plot, max_absorbance_plot)

    plt.grid(True, linestyle='--', alpha=0.7)

# Create interactive widgets
molar_absorptivity_widget = widgets.FloatText(
    value=1.0,
    min=0.01,
    description='Molar Absorptivity (ε):',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

pathlength_widget = widgets.FloatText(
    value=1.0,
    min=0.1,
    description='Pathlength (cm):',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

concentration_widget = widgets.FloatText(
    value=1.0,
    min=0.001,
    description='Concentration (M):',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

marker_wavelength_widget = widgets.IntSlider(
    value=350,
    min=200,
    max=1200,
    step=1,
    description='Marker Wavelength (nm):',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d',
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

marker_wavelength_text = widgets.IntText(
    value=350,
    description='(Enter Value):',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

widgets.jslink((marker_wavelength_widget, 'value'), (marker_wavelength_text, 'value'))

min_wavelength_widget = widgets.IntText(
    value=350,
    min=200,
    description='Min Wavelength (nm):',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

max_wavelength_widget = widgets.IntText(
    value=1100,
    max=1200,
    description='Max Wavelength (nm):',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

min_absorbance_widget = widgets.FloatText(
    value=0.0,
    description='Min Absorbance:',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

max_absorbance_widget_y = widgets.FloatText(
    value=1.0,
    description='Max Absorbance:',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

# Group widgets into sections
curve_shape_widgets = widgets.VBox([
    widgets.HTML('<b>Curve Shape Parameters</b>'),
    molar_absorptivity_widget,
    pathlength_widget,
    concentration_widget
])

plot_display_widgets = widgets.VBox([
    widgets.HTML('<b>Plot Display Parameters</b>'),
    widgets.HBox([marker_wavelength_widget, marker_wavelength_text]),
    min_wavelength_widget,
    max_wavelength_widget,
    min_absorbance_widget,
    max_absorbance_widget_y
])

# Combine the control groups into a horizontal box
ui = widgets.HBox([curve_shape_widgets, plot_display_widgets])

# Use widgets.interactive_output to link the widgets to the plotting function
plot_output = widgets.interactive_output(
    plot_copper_sulfate_spectrum,
    {
        'molar_absorptivity_input': molar_absorptivity_widget,
        'min_wavelength_plot': min_wavelength_widget,
        'max_wavelength_plot': max_wavelength_widget,
        'min_absorbance_plot': min_absorbance_widget,
        'max_absorbance_plot': max_absorbance_widget_y,
        'pathlength_input': pathlength_widget,
        'concentration_input': concentration_widget,
        'marker_wavelength_input': marker_wavelength_widget,
        'fixed_lambda_max': widgets.fixed(initial_lambda_max) # Pass the fixed lambda_max
    }
)

# Display the UI (controls) and then the Output widget (plot)
display(ui, plot_output)

Output()

In [15]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML
from scipy.stats import linregress
import random

# Access the global variables from the spectrum plot (defined in cell 8174b8e1)
# These will be populated when plot_copper_sulfate_spectrum is called.
# If the spectrum cell hasn't been run yet, they will be None.
# We need to handle this case gracefully.

def get_absorbance_at_wavelength(target_wavelength):
    global _global_wavelengths_data, _global_absorbance_data

    if _global_wavelengths_data is None or _global_absorbance_data is None:
        # Fallback if spectrum data isn't available yet or cell hasn't been run
        return 0.0

    # Find the absorbance value at the target_wavelength using interpolation
    # np.interp handles cases where target_wavelength is outside the range by clamping
    return np.interp(target_wavelength, _global_wavelengths_data, _global_absorbance_data)


def generate_standard_curve(
    selected_wavelength=350,
    num_samples=5,
    min_conc=0.0,
    max_conc=1.0,
    peak_molar_absorptivity_val=1.0, # From previous widget
    pathlength_val=1.0,              # From previous widget
    fixed_lambda_max_val=800         # From previous global variable
):
    global _global_max_absorbance # Access global max absorbance

    # Validate inputs
    if num_samples < 2:
        num_samples = 2
    if min_conc >= max_conc:
        min_conc = 0.0
        max_conc = 1.0
    if max_conc <= 0:
        max_conc = 1.0

    # Calculate intensity proportion based on the spectrum data
    absorbance_at_selected_lambda = get_absorbance_at_wavelength(selected_wavelength)

    # Ensure _global_max_absorbance is not zero to avoid division by zero.
    # If the spectrum is flat or all zeros, max_absorbance could be 0.
    if _global_max_absorbance > 0:
        intensity_proportion = absorbance_at_selected_lambda / _global_max_absorbance
    else:
        # If no spectrum data or max absorbance is zero, assume lowest intensity proportion
        intensity_proportion = 0.0

    # Clamp intensity_proportion to [0, 1]
    intensity_proportion = np.clip(intensity_proportion, 0, 1)

    # --- Dynamically adjust noise level based on intensity_proportion ---
    # More noise when intensity_proportion is low (far from lambda_max)
    # Less noise when intensity_proportion is high (near lambda_max)
    min_base_noise_factor = 0.005 # Increased to ensure a slightly higher minimum noise
    max_base_noise_factor = 0.30  # Increased to allow for more variability at intermediate signal strengths

    # Adjust noise range: increases as intensity_proportion decreases
    # Using a polynomial for potentially faster increase in noise away from peak
    noise_scale_from_proportion = (1 - intensity_proportion)**1.5
    dynamic_noise_std = min_base_noise_factor + (max_base_noise_factor - min_base_noise_factor) * noise_scale_from_proportion

    # Ensure a reasonable range for random noise factor selection
    # The actual noise level factor for a given run is a random value within a dynamic range
    noise_level_factor_min = dynamic_noise_std * 0.5
    noise_level_factor_max = dynamic_noise_std * 1.5

    # Clamp noise_level_factor_min to avoid very low values that might cause R^2=1.0
    # Also clamp max to prevent extremely high noise
    noise_level_factor_min = max(noise_level_factor_min, 0.001)
    noise_level_factor_max = min(noise_level_factor_max, 0.6) # Increased max clamp to accommodate larger max_base_noise_factor

    # --- Dynamically adjust R^2 target range based on intensity_proportion ---
    # R^2 closer to 1.0 when intensity_proportion is high
    # R^2 can be as low as 0.6 when intensity_proportion is low

    target_r2_min_base = 0.6 # R^2 min when intensity_proportion is 0
    target_r2_min_peak = 0.98 # R^2 min when intensity_proportion is 1

    # Linear interpolation for target_r2_min
    target_r2_min = target_r2_min_base + (target_r2_min_peak - target_r2_min_base) * intensity_proportion

    # Clamp target_r2_min to be between 0.6 and 0.98
    target_r2_min = np.clip(target_r2_min, 0.6, 0.98)

    # Define the width of the R^2 range as 0.02 as requested
    r2_range_width = 0.02
    target_r2_max = target_r2_min + r2_range_width

    # Ensure target_r2_max does not exceed a realistic upper bound
    target_r2_max = min(target_r2_max, 0.998)

    # Generate ideal concentrations and absorbances
    concentrations_ideal = np.linspace(min_conc, max_conc, num_samples)

    # Calculate the Gaussian factor at the selected wavelength using the fixed_lambda_max
    bandwidth = 70  # Same bandwidth as used in the spectrum generation
    gaussian_factor = np.exp(-(selected_wavelength - fixed_lambda_max_val)**2 / (2 * bandwidth**2))

    # Calculate effective molar absorptivity at the selected wavelength
    # The baseline offset (0.05 * peak_molar_absorptivity_val) is also applied proportionally to concentration
    effective_molar_absorptivity_at_wavelength = (peak_molar_absorptivity_val * gaussian_factor) + (0.05 * peak_molar_absorptivity_val)

    # Calculate ideal absorbances using Beer-Lambert Law: A = εbc
    absorbances_ideal = effective_molar_absorptivity_at_wavelength * pathlength_val * concentrations_ideal

    # --- Add (0,0) data point ---
    concentrations = np.insert(concentrations_ideal, 0, 0.0)
    absorbances_with_zero = np.insert(absorbances_ideal, 0, 0.0)

    # --- Iteratively add variability to non-zero data points until R^2 is in desired range ---
    # This loop is designed to find a set of noisy data points that yields an R^2 within the target range.
    # It tries up to max_attempts. If it can't find one, it will use the last generated noisy data.
    attempts = 0
    max_attempts = 500

    # Initialize final_ variables to hold the result of the last iteration, or a successful one
    final_absorbances_noisy = np.copy(absorbances_with_zero)
    final_slope, final_intercept, final_r_squared = 0, 0, 0.0

    while attempts < max_attempts:
        current_absorbances_noisy = np.copy(absorbances_with_zero)

        # Use the dynamically calculated noise_level_factor_min/max for overall noise scaling
        noise_std_factor = random.uniform(noise_level_factor_min, noise_level_factor_max)

        # Calculate the absolute noise standard deviation for this entire curve generation attempt.
        # This makes noise significant even for very low absorbance points when far from lambda_max.
        # Use _global_max_absorbance (peak of the entire spectrum) for scaling to ensure noise level
        # is relevant across all selected wavelengths, even if current ideal absorbances are low.
        # Add a floor of 0.01 to _global_max_absorbance to ensure some noise can always be generated.
        noise_reference_value = max(0.01, _global_max_absorbance)
        noise_std_for_this_attempt = noise_reference_value * noise_std_factor

        for i in range(1, len(current_absorbances_noisy)): # Skip the (0,0) point (index 0)
            base_absorbance = absorbances_with_zero[i]

            # Generate noise for this point using the attempt-wide noise_std
            noise = np.random.normal(0, noise_std_for_this_attempt)
            noisy_absorbance_candidate = base_absorbance + noise

            # Apply non-negativity constraint.
            # If the ideal absorbance was positive but the noisy value went negative,
            # set it to a very small random positive value to ensure variability
            # and prevent multiple points from becoming exactly zero.
            if base_absorbance > 0 and noisy_absorbance_candidate < 0:
                current_absorbances_noisy[i] = np.random.uniform(1e-5, 5e-5) # Small random positive value
            elif noisy_absorbance_candidate < 0:
                current_absorbances_noisy[i] = 0.0 # If ideal was 0 or negative (shouldn't happen for i>0), clamp to 0
            else:
                current_absorbances_noisy[i] = noisy_absorbance_candidate

        # Calculate regression parameters for the current noisy data
        current_slope, current_intercept, current_r_value, _, _ = 0, 0, 0, 0, 0
        current_r_squared = 0.0

        if len(np.unique(concentrations)) >= 2 and len(np.unique(current_absorbances_noisy)) >= 2:
            current_slope, current_intercept, current_r_value, _, _ = linregress(concentrations, current_absorbances_noisy)
            current_r_squared = current_r_value**2

        # Always update the 'final' variables with the results of the current iteration
        final_absorbances_noisy = current_absorbances_noisy
        final_slope = current_slope
        final_intercept = current_intercept
        final_r_squared = current_r_squared

        # Check if the target R^2 is met
        if target_r2_min <= final_r_squared <= target_r2_max:
            break # Exit loop if R^2 is in the desired range

        attempts += 1

    # After the loop, use the 'final_' variables for plotting and text display
    absorbances_noisy = final_absorbances_noisy
    slope = final_slope
    intercept = final_intercept
    r_squared = final_r_squared

    # Create the plot
    plt.figure(figsize=(5.0, 3.0))

    # Plot noisy data points
    plt.plot(concentrations, absorbances_noisy, 'o', color='green', label='Data Points')

    # Plot regression line
    plt.plot(concentrations, slope * concentrations + intercept, color='red', linestyle='--', label='Linear Regression')

    plt.title(rf'Standard Curve at {selected_wavelength} nm ($\lambda_{{max}}$: {fixed_lambda_max_val} nm)')
    plt.xlabel('Concentration (M)')
    plt.ylabel('Absorbance')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.ylim(bottom=0) # Ensure y-axis starts at 0
    plt.xlim(left=0)  # Ensure x-axis starts at 0

    # --- Display regression equation and correlation coefficient ---
    eq_text = f'y = {slope:.3f}x + {intercept:.3f}'
    r2_text = f'R$^2$ = {r_squared:.3f} (Target: {target_r2_min:.3f}-{target_r2_max:.3f})' # Show target R^2 range

    # Position text based on plot limits
    x_range = plt.xlim()[1] - plt.xlim()[0]
    y_range = plt.ylim()[1] - plt.ylim()[0]
    plt.text(plt.xlim()[0] + 0.05 * x_range, plt.ylim()[1] - 0.1 * y_range, eq_text, color='red', fontsize=10)
    plt.text(plt.xlim()[0] + 0.05 * x_range, plt.ylim()[1] - 0.18 * y_range, r2_text, color='red', fontsize=10)

    plt.legend()
    plt.show()

# Create widgets for the standard curve
sc_wavelength_widget = widgets.IntSlider(
    value=marker_wavelength_widget.value, # Default to the marker wavelength from the first plot
    min=200,
    max=1200,
    step=1,
    description='Wavelength (nm):',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d',
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

sc_num_samples_widget = widgets.IntSlider(
    value=5,
    min=2,
    max=20,
    step=1,
    description='Number of Samples:',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d',
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

sc_min_concentration_widget = widgets.FloatText(
    value=0.0,
    min=0.0,
    description='Min Conc. (M):',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

sc_max_concentration_widget = widgets.FloatText(
    value=1.0,
    min=0.01,
    description='Max Conc. (M):',
    disabled=False,
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

# Group standard curve widgets
standard_curve_widgets = widgets.VBox([
    widgets.HTML('<b>Standard Curve Parameters</b>'),
    sc_wavelength_widget,
    sc_num_samples_widget,
    sc_min_concentration_widget,
    sc_max_concentration_widget
])

# Link to the generate_standard_curve function
standard_curve_output = widgets.interactive_output(
    generate_standard_curve,
    {
        'selected_wavelength': sc_wavelength_widget,
        'num_samples': sc_num_samples_widget,
        'min_conc': sc_min_concentration_widget,
        'max_conc': sc_max_concentration_widget,
        'peak_molar_absorptivity_val': widgets.fixed(molar_absorptivity_widget.value),
        'pathlength_val': widgets.fixed(pathlength_widget.value),
        'fixed_lambda_max_val': widgets.fixed(initial_lambda_max)
    }
)

# Display the standard curve UI and output
display(standard_curve_widgets, standard_curve_output)

Output()